In [27]:
from typing import Any

import torch
from torch import nn
import torch_geometric
from torch_geometric.nn import GCNConv
from torch_geometric.datasets import KarateClub
from torch_geometric.utils import negative_sampling
import torch.nn.functional as F


In [28]:
data = KarateClub()

In [34]:
dir(data)

['__add__',
 '__annotate_func__',
 '__annotations_cache__',
 '__class__',
 '__class_getitem__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__orig_bases__',
 '__parameters__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_data',
 '_data_list',
 '_download',
 '_indices',
 '_infer_num_classes',
 '_process',
 'collate',
 'copy',
 'cpu',
 'cuda',
 'data',
 'download',
 'force_reload',
 'get',
 'get_summary',
 'has_download',
 'has_process',
 'index_select',
 'indices',
 'len',
 'load',
 'log',
 'num_classes',
 'num_edge_features',
 'num_features',
 'num_node_features',
 'pre_filter',
 'pre_transform',
 'pr

In [29]:
len(data)

1

In [35]:
data[0]

Data(x=[34, 34], edge_index=[2, 156], y=[34], train_mask=[34])

In [36]:
ds = data[0]

In [37]:
ds

Data(x=[34, 34], edge_index=[2, 156], y=[34], train_mask=[34])

In [38]:
ds.new_attribute = torch.tensor([1,2,3])

In [39]:
ds

Data(x=[34, 34], edge_index=[2, 156], y=[34], train_mask=[34], new_attribute=[3])

In [40]:
class GNNEncoder(nn.Module):
    def __init__(self, in_channels:int,out_channels:int):
        super().__init__()
        self.conv1 = GCNConv(in_channels,16)
        self.conv2 = GCNConv(16,out_channels)

    def forward(self,data):
        x,edge_index = data.x,data.edge_index
        x = F.relu(self.conv1(x,edge_index)) # GCNConv see assignment to see how message passing is done
        x = self.conv2(x,edge_index)
        return x # Node embeddings




In [41]:
data.num_features

34

In [42]:
ds.x

tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [0., 1., 0.,  ..., 0., 0., 0.],
        [0., 0., 1.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 1., 0., 0.],
        [0., 0., 0.,  ..., 0., 1., 0.],
        [0., 0., 0.,  ..., 0., 0., 1.]])

In [43]:
ds.edge_index

tensor([[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  1,
          1,  1,  1,  1,  1,  1,  1,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  3,
          3,  3,  3,  3,  3,  4,  4,  4,  5,  5,  5,  5,  6,  6,  6,  6,  7,  7,
          7,  7,  8,  8,  8,  8,  8,  9,  9, 10, 10, 10, 11, 12, 12, 13, 13, 13,
         13, 13, 14, 14, 15, 15, 16, 16, 17, 17, 18, 18, 19, 19, 19, 20, 20, 21,
         21, 22, 22, 23, 23, 23, 23, 23, 24, 24, 24, 25, 25, 25, 26, 26, 27, 27,
         27, 27, 28, 28, 28, 29, 29, 29, 29, 30, 30, 30, 30, 31, 31, 31, 31, 31,
         31, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 33, 33, 33, 33, 33,
         33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33],
        [ 1,  2,  3,  4,  5,  6,  7,  8, 10, 11, 12, 13, 17, 19, 21, 31,  0,  2,
          3,  7, 13, 17, 19, 21, 30,  0,  1,  3,  7,  8,  9, 13, 27, 28, 32,  0,
          1,  2,  7, 12, 13,  0,  6, 10,  0,  6, 10, 16,  0,  4,  5, 16,  0,  1,
          2,  3,  0,  2, 30, 32, 33,  2, 33,  0,  4

In [70]:

model = GNNEncoder(data.num_features,8)

In [71]:
model

GNNEncoder(
  (conv1): GCNConv(34, 16)
  (conv2): GCNConv(16, 8)
)

In [46]:
edge_predictions=(ds.edge_index[0]*ds.edge_index[1]).sum(dim=-1)

In [47]:
z_source = ds.edge_index
z_target = ds.x[1]

In [48]:
z_source.shape

torch.Size([2, 156])

In [57]:
def predict_edge_between_nodes(edge_index):
    source = edge_index[0]
    target = edge_index[1]
    return (source*target).sum(dim=-1).float()

## Training model

In [58]:
optim = torch.optim.Adam(model.parameters(),lr=.01)


In [72]:
# Embeddings

z = model(data)

# Positive edges
model.train()
for epoch in range(100):
    positive_prediction = predict_edge_between_nodes(ds.edge_index)

    # Negative Sampling why?
    #\# 3\. Negative Edges (Random fake edges to teach the model what a "0" looks like)
    negative_edge_index =negative_sampling(edge_index=ds.edge_index,num_nodes=ds.num_nodes,num_neg_samples=ds.edge_index.size(1))

    negative_prediction = predict_edge_between_nodes(negative_edge_index)

    # positive loss
    pos_loss = F.binary_cross_entropy_with_logits(positive_prediction,torch.ones_like(positive_prediction))
    print(pos_loss)
    # Negative loss
    neg_loss = F.binary_cross_entropy_with_logits(negative_prediction,torch.zeros_like(negative_prediction))

    #Total loss
    loss = pos_loss + neg_loss
    if loss.requires_grad:
        print("Yes")
        loss.backward()
    optim.zero_grad()
    optim.step()


tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)

In [73]:
loss

tensor(39747.)

In [53]:
negative_edge_index =negative_sampling(edge_index=ds.edge_index,num_nodes=ds.num_nodes,num_neg_samples=ds.edge_index.size(1))
negative_prediction=predict_edge_between_nodes(negative_edge_index)
print(type(negative_prediction))


<class 'torch.Tensor'>


In [62]:
negative_prediction

tensor(36298.)

In graph-level or node-level tasks, you use the GNN to output a classification directly. For edge prediction, the GNN is only the first half of the architecture (the Encoder). You need a second half (the Decoder) to evaluate the edges.
1. The Encoder (GNN): First, you run a standard GNN (like GCNConv) over the graph to generate node embeddings.
* Node $u$ gets embedding vector $z_u$
* Node $v$ gets embedding vector $z_v$
2. The Decoder (Similarity metric): To predict an edge between $u$ and $v$, you combine their embeddings to generate a single score. The most common method is the Dot Product: $$s_{u,v} = z_u \cdot z_v$$ Because the GNN puts structurally similar nodes close together in the vector space, a high dot product mathematically proves they are highly aligned.
3. The Probability: We pass that raw score through a Sigmoid function ($\sigma$) to squash it between 0 and 1: $$\hat{y}_{u,v} = \sigma(s_{u,v})$$
4. The Loss (BCE): Since an edge either exists (1) or doesn't (0), this is treated as binary classification using Binary Cross Entropy loss.

In [54]:
negative_prediction.dtype

torch.int64

In [63]:
torch.zeros_like(negative_prediction)

tensor(0.)

In [60]:
predict_edge_between_nodes(negative_edge_index)


tensor(36298.)